# 03 — Prepare & Export

Package the cleaned `films_adjusted` table as the published dataset: CSV
(+ Excel/Parquet) with a plain-English codebook for every column.

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from src.ingest import load_config
from src.clean_quality import get_connection
from src.prepare import package_dataset

cfg = load_config('config.yaml')
con = get_connection(cfg)
films = con.execute('SELECT * FROM films_adjusted ORDER BY adjusted_gross DESC').df()
print(films.shape)

## Codebook — a plain-English description for every column

In [ ]:
codebook = {
    'rank_adjusted':     'Rank by inflation-adjusted domestic gross (1 = highest).',
    'title':             'Film title.',
    'adjusted_gross':    'Domestic lifetime gross adjusted to 2022 dollars via ticket-price inflation (USD).',
    'nominal_gross':     'Domestic lifetime gross in year-of-release dollars, as originally reported (USD).',
    'est_tickets':       'Estimated number of tickets sold over the film lifetime (Box Office Mojo estimate).',
    'release_year':      'Year of the film original theatrical release.',
    'decade':            'Release decade (release_year rounded down to the nearest 10).',
    'inflation_multiple':'adjusted_gross / nominal_gross - how many times its original take the adjusted figure represents.',
}

notes = '''
Source: Box Office Mojo, Top Lifetime Adjusted Grosses (domestic, US/Canada), adjusted to 2022 dollars.
URL: https://www.boxofficemojo.com/chart/top_lifetime_gross_adjusted/?adjust_gross_to=2022

Method: adjusted gross = estimated tickets sold x the 2022 average ticket price (ticket-price inflation,
NOT CPI). Domestic only. A film lifetime total includes re-release grosses, which inflates some classics.
Nominal grosses are in year-of-release dollars and are not comparable across eras without the adjustment.
'''
written = package_dataset(films, cfg, name='highest_grossing_films_v1', codebook=codebook, notes=notes)
written

---
**Next:** `04-viz.ipynb` (matplotlib exploration) and `06-viz-social.ipynb` (the social chart).

## Cleanup

In [ ]:
con.close()
print('connection closed')